In [ ]:
"""
Simulation Log Parser + Pillar-Targeting Distribution Plot
===========================================================
Automatically discovers every versioned run folder (v1, v2, v3, …).

Directory layout
────────────────
INPUT  (one folder per run):
    <PROJECT_ROOT>/SimulationLogs/gemini_flash/ZeroShot/stateful/vision-4/
        AgentsLogging/simulation_log/unstructured/v<N>/step_*.txt

OUTPUT (mirrored, one folder per run):
    <PROJECT_ROOT>/SimulationLogs/gemini_flash/ZeroShot/stateful/vision-4/
        AgentsLogging/simulation_log/structured/activists_logs/v<N>/
            parsed_simulation_log.csv
            pillar_targeting_distribution.png

Usage
─────
    python parse_and_plot_logs.py
"""

import ast
import csv
import glob
import json
import os
import re
from collections import Counter, defaultdict
from pathlib import Path

# ─────────────────────────────────────────────────────────────────────────────
# DIRECTORY CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

PROJECT_ROOT = Path.cwd().parent

_LOGS_BASE = (
    PROJECT_ROOT
    / "SimulationLogs" / "gemini_flash" / "ZeroShot"
    / "stateful"       / "vision-4"
    / "AgentsLogging"  / "simulation_log"
)

# Base directories — NO v* here; version suffix is appended at runtime
DIRS = {
    # Root that contains v1/, v2/, v3/, … sub-directories of raw step logs
    "agent_dir":        _LOGS_BASE / "unstructured",

    # Root where structured output folders v1/, v2/, … will be created
    "agent_output_dir": _LOGS_BASE / "structured" / "activists_logs",
}

# File names written inside each per-version output folder
OUTPUT_CSV  = "activists_logs_v*.csv"
OUTPUT_PLOT = "pillar_targeting_distribution.png"


# ─────────────────────────────────────────────────────────────────────────────
# REGEX PATTERNS
# ─────────────────────────────────────────────────────────────────────────────

FIELD_PATTERN = re.compile(
    r"""
    (?P<key>[a-zA-Z_]+)
    =
    (?P<value>
        "(?:[^"\\]|\\.)*"       # double-quoted string
        |'(?:[^'\\]|\\.)*'      # single-quoted string
        |\([^)]*\)               # tuple  (x, y)
        |[^;{]+                  # plain value
    )
    """,
    re.VERBOSE,
)

CONTEXT_PATTERN = re.compile(r"context=(\{.*\})$", re.DOTALL)
MEMORY_PATTERN  = re.compile(r"memory=(\{.*?\}(?:\}|\s*\}))", re.DOTALL)


# ─────────────────────────────────────────────────────────────────────────────
# PARSING HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def _clean(v: str) -> str:
    v = v.strip()
    if (v.startswith('"') and v.endswith('"')) or \
       (v.startswith("'") and v.endswith("'")):
        return v[1:-1]
    return v


def _parse_tuple(v: str):
    try:
        return ast.literal_eval(v.strip())
    except Exception:
        return v.strip()


def parse_line(line: str) -> dict | None:
    line = line.strip()
    if not line:
        return None

    record = {}

    # 1. Peel off context={...} JSON blob
    ctx_match = CONTEXT_PATTERN.search(line)
    if ctx_match:
        try:
            record["context"] = json.loads(ctx_match.group(1))
        except json.JSONDecodeError:
            record["context_raw"] = ctx_match.group(1)
        line = line[: ctx_match.start()]

    # 2. Peel off memory={...} Python-dict blob
    mem_match = MEMORY_PATTERN.search(line)
    if mem_match:
        try:
            record["memory"] = ast.literal_eval(mem_match.group(1))
        except Exception:
            record["memory_raw"] = mem_match.group(1)
        line = line[: mem_match.start()]

    # 3. Parse remaining key=value fields
    for m in FIELD_PATTERN.finditer(line):
        record[m.group("key")] = _clean(m.group("value"))

    if not record:
        return None

    # 4. Type coercions
    for k in ("Step", "agent_id", "response_tokens"):
        if k in record:
            try:
                record[k] = int(record[k])
            except (ValueError, TypeError):
                pass

    if "elapsed" in record:
        try:
            record["elapsed"] = float(str(record["elapsed"]).rstrip("s"))
        except (ValueError, TypeError):
            pass

    for k in ("pos", "from_pos", "to_pos"):
        if k in record:
            record[k] = _parse_tuple(record[k])

    return record


def parse_step_file(filepath: str, step_number: int) -> list[dict]:
    records = []
    with open(filepath, "r", encoding="utf-8", errors="replace") as fh:
        for line in fh:
            rec = parse_line(line)
            if rec:
                rec.setdefault("Step", step_number)
                records.append(rec)
    return records


def _step_num(path: str) -> int:
    m = re.search(r"step_(\d+)\.txt$", path, re.IGNORECASE)
    return int(m.group(1)) if m else 0


def parse_all_steps(unstructured_dir: Path) -> list[dict]:
    """Parse every step_*.txt inside *unstructured_dir*."""
    files = sorted(
        glob.glob(str(unstructured_dir / "step_*.txt")),
        key=_step_num,
    )
    if not files:
        raise FileNotFoundError(
            f"No step_*.txt files found in:\n  {unstructured_dir}"
        )
    print(f"  Found {len(files)} step files — parsing …")
    records = []
    for fp in files:
        records.extend(parse_step_file(fp, _step_num(fp)))
    print(f"  Parsed {len(records):,} log records.")
    return records


# ─────────────────────────────────────────────────────────────────────────────
# CSV EXPORT
# ─────────────────────────────────────────────────────────────────────────────

FLAT_COLUMNS = [
    "Step", "agent_type", "agent_id", "event",
    "pos", "from_pos", "to_pos",
    "elapsed", "response_tokens",
    "target_agent", "target_pillar_inst", "primary_attribute",
    "citizen_cta", "coordination_state", "derived_strategy",
    "target_pillar_institution",
]


def _flatten(rec: dict) -> dict:
    flat = {col: rec.get(col, "") for col in FLAT_COLUMNS}

    ctx = rec.get("context", {})
    if isinstance(ctx, dict):
        flat["coordination_state"]       = flat["coordination_state"]       or ctx.get("coordination_state", "")
        flat["derived_strategy"]         = flat["derived_strategy"]         or ctx.get("derived_strategy", "")
        flat["target_pillar_institution"]= flat["target_pillar_institution"] or ctx.get("target_pillar_institution", "")

    # Stringify any leftover complex types
    for k in FLAT_COLUMNS:
        if isinstance(flat[k], (list, tuple, dict)):
            flat[k] = str(flat[k])

    return flat


def write_csv(records: list[dict], output_path: Path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=FLAT_COLUMNS, extrasaction="ignore")
        writer.writeheader()
        for rec in records:
            writer.writerow(_flatten(rec))
    print(f"  CSV  → {output_path}")


# ─────────────────────────────────────────────────────────────────────────────
# PLOT
# ─────────────────────────────────────────────────────────────────────────────

def _get_pillar(rec: dict) -> str:
    """Return the targeted pillar name from a record (checks multiple fields)."""
    pillar = rec.get("target_pillar_inst", "").strip()
    if not pillar:
        ctx = rec.get("context", {})
        if isinstance(ctx, dict):
            pillar = ctx.get("target_pillar_institution", "").strip()
    return pillar


def plot_pillar_targeting(records: list[dict], output_path: Path, version_label: str):
    try:
        import matplotlib.pyplot as plt
        import matplotlib.ticker as ticker
    except ImportError:
        print("  matplotlib not installed — pip install matplotlib")
        return

    step_total     : dict[int, int]             = defaultdict(int)
    step_breakdown : dict[int, dict[str, int]]  = defaultdict(lambda: defaultdict(int))

    for rec in records:
        if rec.get("agent_type") != "ActivistAgent":
            continue
        pillar = _get_pillar(rec)
        if pillar:
            s = rec.get("Step", 0)
            step_total[s] += 1
            step_breakdown[s][pillar] += 1

    if not step_total:
        print("  No pillar-targeting events found — skipping plot.")
        return

    steps   = sorted(step_total)
    totals  = [step_total[s] for s in steps]
    pillars = sorted({p for d in step_breakdown.values() for p in d})

    palette = [
        "#4e8ef7", "#f76e6e", "#6ef7b4", "#f7c06e", "#c06ef7",
        "#6ec9f7", "#f76ec0", "#c9f76e", "#f7906e", "#6e90f7",
    ]

    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(16, 9),
        gridspec_kw={"height_ratios": [3, 1.2]},
        sharex=True,
    )
    fig.patch.set_facecolor("#0f1117")
    for ax in (ax1, ax2):
        ax.set_facecolor("#1a1d27")
        ax.tick_params(colors="#c9cdd6")
        for spine in ax.spines.values():
            spine.set_edgecolor("#2e3142")

    # Stacked bars
    bottom = [0] * len(steps)
    for i, pillar in enumerate(pillars):
        vals = [step_breakdown[s].get(pillar, 0) for s in steps]
        ax1.bar(steps, vals, bottom=bottom,
                label=pillar, color=palette[i % len(palette)],
                alpha=0.85, width=0.85)
        bottom = [b + v for b, v in zip(bottom, vals)]

    ax1.set_ylabel("# Pillar-Targeting Events", color="#c9cdd6", fontsize=11)
    ax1.set_title(
        f"Activist → Pillar Targeting Distribution  [{version_label}]",
        color="#ffffff", fontsize=14, pad=12, fontweight="bold",
    )
    ax1.legend(loc="upper right", framealpha=0.25, facecolor="#1a1d27",
               edgecolor="#2e3142", labelcolor="#c9cdd6", fontsize=8)
    ax1.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax1.grid(axis="y", color="#2e3142", linewidth=0.6)

    # Total line
    ax2.plot(steps, totals, color="#4e8ef7", linewidth=1.8,
             marker="o", markersize=2.5)
    ax2.fill_between(steps, totals, alpha=0.15, color="#4e8ef7")
    ax2.set_xlabel("Step", color="#c9cdd6", fontsize=11)
    ax2.set_ylabel("Total", color="#c9cdd6", fontsize=10)
    ax2.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax2.grid(axis="y", color="#2e3142", linewidth=0.6)

    plt.tight_layout(h_pad=0.5)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(output_path, dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"  Plot → {output_path}")


# ─────────────────────────────────────────────────────────────────────────────
# SUMMARY STATS
# ─────────────────────────────────────────────────────────────────────────────

def print_summary(records: list[dict]):
    counts: Counter = Counter()
    for rec in records:
        if rec.get("agent_type") == "ActivistAgent":
            pillar = _get_pillar(rec)
            if pillar:
                counts[pillar] += 1
    if not counts:
        print("  (no pillar-targeting events)")
        return
    width = max(len(p) for p in counts)
    for pillar, cnt in counts.most_common():
        print(f"    {pillar:<{width}}  {cnt:>6}")
    print(f"    {'TOTAL':<{width}}  {sum(counts.values()):>6}")


# ─────────────────────────────────────────────────────────────────────────────
# DISCOVER VERSIONED RUNS
# ─────────────────────────────────────────────────────────────────────────────

def discover_versions() -> list[tuple[Path, Path, str]]:
    """
    Scans DIRS["agent_dir"] for v<N> sub-directories and returns a list of
        (input_dir, output_dir, version_label)
    sorted numerically (v1, v2, v3, …).

    input_dir  — …/unstructured/v<N>     (must exist, contains step_*.txt files)
    output_dir — …/activists_logs/v<N>   (created automatically if absent)
    """
    input_base  = DIRS["agent_dir"]         # …/unstructured
    output_base = DIRS["agent_output_dir"]  # …/activists_logs

    def _version_num(p: str) -> int:
        m = re.search(r"[/\\]v(\d+)$", p)
        return int(m.group(1)) if m else 0

    version_dirs = sorted(
        [p for p in glob.glob(str(input_base / "v*")) if os.path.isdir(p)],
        key=_version_num,
    )

    if not version_dirs:
        raise FileNotFoundError(
            f"No versioned run folders (v1, v2, …) found under:\n  {input_base}\n"
            "Check that PROJECT_ROOT points to the right place."
        )

    runs = []
    for vdir in version_dirs:
        vname = Path(vdir).name           # "v1", "v2", …
        runs.append((Path(vdir), output_base, vname))

    return runs


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

def process_version(input_dir: Path, output_dir: Path, version_label: str):
    print(f"\n{'─'*60}")
    print(f"  Run: {version_label}")
    print(f"  Input  : {input_dir}")
    print(f"  Output : {output_dir}")
    print(f"{'─'*60}")

    records = parse_all_steps(input_dir)

    write_csv(records, output_dir / OUTPUT_CSV)
    plot_pillar_targeting(records, output_dir / OUTPUT_PLOT, version_label)

    print("  Pillar-targeting summary:")
    print_summary(records)


if __name__ == "__main__":
    runs = discover_versions()
    print(f"Discovered {len(runs)} version(s): "
          f"{', '.join(v for _, _, v in runs)}")

    for input_dir, output_dir, version_label in runs:
        process_version(input_dir, output_dir, version_label)

    print("\n✓ All runs processed.")

Discovered 5 version(s): v1, v2, v3, v4, v5

────────────────────────────────────────────────────────────
  Run: v1
  Input  : c:\Users\fmahnoor1\Documents\Research-FL\ABM-Repos\New Approach\pyabm-refactor\SimulationLogs\gemini_flash\ZeroShot\stateful\vision-4\AgentsLogging\simulation_log\unstructured\v1
  Output : c:\Users\fmahnoor1\Documents\Research-FL\ABM-Repos\New Approach\pyabm-refactor\SimulationLogs\gemini_flash\ZeroShot\stateful\vision-4\AgentsLogging\simulation_log\structured\activists_logs\v1
────────────────────────────────────────────────────────────
  Found 96 step files — parsing …
  Parsed 1,520 log records.
  CSV  → c:\Users\fmahnoor1\Documents\Research-FL\ABM-Repos\New Approach\pyabm-refactor\SimulationLogs\gemini_flash\ZeroShot\stateful\vision-4\AgentsLogging\simulation_log\structured\activists_logs\v1\parsed_simulation_log.csv
  Plot → c:\Users\fmahnoor1\Documents\Research-FL\ABM-Repos\New Approach\pyabm-refactor\SimulationLogs\gemini_flash\ZeroShot\stateful\vision-

In [ ]:
"""
Simulation Log Parser + Pillar-Targeting Distribution Plot
===========================================================
Automatically discovers every versioned run folder (v1, v2, v3, …).

Directory layout
────────────────
INPUT  (one folder per run):
    <PROJECT_ROOT>/SimulationLogs/gemini_flash/ZeroShot/stateful/vision-4/
        AgentsLogging/simulation_log/unstructured/v<N>/step_*.txt

OUTPUT (mirrored, one folder per run):
    <PROJECT_ROOT>/SimulationLogs/gemini_flash/ZeroShot/stateful/vision-4/
        AgentsLogging/simulation_log/structured/activists_logs/v<N>/
            parsed_simulation_log.csv
            pillar_targeting_distribution.png

Usage
─────
    python parse_and_plot_logs.py
"""

import ast
import csv
import glob
import json
import os
import re
from collections import Counter, defaultdict
from pathlib import Path

# ─────────────────────────────────────────────────────────────────────────────
# DIRECTORY CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

PROJECT_ROOT = Path.cwd().parent

_LOGS_BASE = (
    PROJECT_ROOT
    / "SimulationLogs" / "gemini_flash" / "ZeroShot"
    / "stateful"       / "vision-4"
    / "AgentsLogging"  / "simulation_log"
)

# Base directories — NO v* here; version suffix is appended at runtime
DIRS = {
    # Root that contains v1/, v2/, v3/, … sub-directories of raw step logs
    "agent_dir":        _LOGS_BASE / "unstructured",

    # Root where structured output folders v1/, v2/, … will be created
    "agent_output_dir": _LOGS_BASE / "structured" / "activists_logs",
}

# File name templates — {v} is replaced with the version label at runtime
# All files land directly in DIRS["agent_output_dir"] with no sub-folders.
OUTPUT_CSV_TPL  = "activists_logs_{v}.csv"
OUTPUT_PLOT_TPL = "pillar_targeting_distribution_{v}.png"


# ─────────────────────────────────────────────────────────────────────────────
# REGEX PATTERNS
# ─────────────────────────────────────────────────────────────────────────────

FIELD_PATTERN = re.compile(
    r"""
    (?P<key>[a-zA-Z_]+)
    =
    (?P<value>
        "(?:[^"\\]|\\.)*"       # double-quoted string
        |'(?:[^'\\]|\\.)*'      # single-quoted string
        |\([^)]*\)               # tuple  (x, y)
        |[^;{]+                  # plain value
    )
    """,
    re.VERBOSE,
)

CONTEXT_PATTERN = re.compile(r"context=(\{.*\})$", re.DOTALL)
MEMORY_PATTERN  = re.compile(r"memory=(\{.*?\}(?:\}|\s*\}))", re.DOTALL)


# ─────────────────────────────────────────────────────────────────────────────
# PARSING HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def _clean(v: str) -> str:
    v = v.strip()
    if (v.startswith('"') and v.endswith('"')) or \
       (v.startswith("'") and v.endswith("'")):
        return v[1:-1]
    return v


def _parse_tuple(v: str):
    try:
        return ast.literal_eval(v.strip())
    except Exception:
        return v.strip()


def parse_line(line: str) -> dict | None:
    line = line.strip()
    if not line:
        return None

    record = {}

    # 1. Peel off context={...} JSON blob
    ctx_match = CONTEXT_PATTERN.search(line)
    if ctx_match:
        try:
            record["context"] = json.loads(ctx_match.group(1))
        except json.JSONDecodeError:
            record["context_raw"] = ctx_match.group(1)
        line = line[: ctx_match.start()]

    # 2. Peel off memory={...} Python-dict blob
    mem_match = MEMORY_PATTERN.search(line)
    if mem_match:
        try:
            record["memory"] = ast.literal_eval(mem_match.group(1))
        except Exception:
            record["memory_raw"] = mem_match.group(1)
        line = line[: mem_match.start()]

    # 3. Parse remaining key=value fields
    for m in FIELD_PATTERN.finditer(line):
        record[m.group("key")] = _clean(m.group("value"))

    if not record:
        return None

    # 4. Type coercions
    for k in ("Step", "agent_id", "response_tokens"):
        if k in record:
            try:
                record[k] = int(record[k])
            except (ValueError, TypeError):
                pass

    if "elapsed" in record:
        try:
            record["elapsed"] = float(str(record["elapsed"]).rstrip("s"))
        except (ValueError, TypeError):
            pass

    for k in ("pos", "from_pos", "to_pos"):
        if k in record:
            record[k] = _parse_tuple(record[k])

    return record


def parse_step_file(filepath: str, step_number: int) -> list[dict]:
    records = []
    with open(filepath, "r", encoding="utf-8", errors="replace") as fh:
        for line in fh:
            rec = parse_line(line)
            if rec:
                rec.setdefault("Step", step_number)
                records.append(rec)
    return records


def _step_num(path: str) -> int:
    m = re.search(r"step_(\d+)\.txt$", path, re.IGNORECASE)
    return int(m.group(1)) if m else 0


def parse_all_steps(unstructured_dir: Path) -> list[dict]:
    """Parse every step_*.txt inside *unstructured_dir*."""
    files = sorted(
        glob.glob(str(unstructured_dir / "step_*.txt")),
        key=_step_num,
    )
    if not files:
        raise FileNotFoundError(
            f"No step_*.txt files found in:\n  {unstructured_dir}"
        )
    print(f"  Found {len(files)} step files — parsing …")
    records = []
    for fp in files:
        records.extend(parse_step_file(fp, _step_num(fp)))
    print(f"  Parsed {len(records):,} log records.")
    return records


# ─────────────────────────────────────────────────────────────────────────────
# CSV EXPORT
# ─────────────────────────────────────────────────────────────────────────────

FLAT_COLUMNS = [
    "Step", "agent_type", "agent_id", "event",
    "pos", "from_pos", "to_pos",
    "elapsed", "response_tokens",
    "target_agent", "target_pillar_inst", "primary_attribute",
    "citizen_cta", "coordination_state", "derived_strategy",
    "target_pillar_institution",
]


def _flatten(rec: dict) -> dict:
    flat = {col: rec.get(col, "") for col in FLAT_COLUMNS}

    ctx = rec.get("context", {})
    if isinstance(ctx, dict):
        flat["coordination_state"]       = flat["coordination_state"]       or ctx.get("coordination_state", "")
        flat["derived_strategy"]         = flat["derived_strategy"]         or ctx.get("derived_strategy", "")
        flat["target_pillar_institution"]= flat["target_pillar_institution"] or ctx.get("target_pillar_institution", "")

    # Stringify any leftover complex types
    for k in FLAT_COLUMNS:
        if isinstance(flat[k], (list, tuple, dict)):
            flat[k] = str(flat[k])

    return flat


def write_csv(records: list[dict], output_path: Path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=FLAT_COLUMNS, extrasaction="ignore")
        writer.writeheader()
        for rec in records:
            writer.writerow(_flatten(rec))
    print(f"  CSV  → {output_path}")


# ─────────────────────────────────────────────────────────────────────────────
# PLOT
# ─────────────────────────────────────────────────────────────────────────────

def _get_pillar(rec: dict) -> str:
    """Return the targeted pillar name from a record (checks multiple fields)."""
    pillar = rec.get("target_pillar_inst", "").strip()
    if not pillar:
        ctx = rec.get("context", {})
        if isinstance(ctx, dict):
            pillar = ctx.get("target_pillar_institution", "").strip()
    return pillar


def plot_pillar_targeting(records: list[dict], output_path: Path, version_label: str):
    try:
        import matplotlib.pyplot as plt
        import matplotlib.ticker as ticker
    except ImportError:
        print("  matplotlib not installed — pip install matplotlib")
        return

    step_total     : dict[int, int]             = defaultdict(int)
    step_breakdown : dict[int, dict[str, int]]  = defaultdict(lambda: defaultdict(int))

    for rec in records:
        if rec.get("agent_type") != "ActivistAgent":
            continue
        if rec.get("target_agent") != "pillars":   # ← add this
            continue
        pillar = _get_pillar(rec)
        if pillar:
            s = rec.get("Step", 0)
            step_total[s] += 1
            step_breakdown[s][pillar] += 1

    if not step_total:
        print("  No pillar-targeting events found — skipping plot.")
        return

    steps   = sorted(step_total)
    totals  = [step_total[s] for s in steps]
    pillars = sorted({p for d in step_breakdown.values() for p in d})

    palette = [
        "#4e8ef7", "#f76e6e", "#6ef7b4", "#f7c06e", "#c06ef7",
        "#6ec9f7", "#f76ec0", "#c9f76e", "#f7906e", "#6e90f7",
    ]

    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(16, 9),
        gridspec_kw={"height_ratios": [3, 1.2]},
        sharex=True,
    )
    fig.patch.set_facecolor("#0f1117")
    for ax in (ax1, ax2):
        ax.set_facecolor("#1a1d27")
        ax.tick_params(colors="#c9cdd6")
        for spine in ax.spines.values():
            spine.set_edgecolor("#2e3142")

    # Stacked bars
    bottom = [0] * len(steps)
    for i, pillar in enumerate(pillars):
        vals = [step_breakdown[s].get(pillar, 0) for s in steps]
        ax1.bar(steps, vals, bottom=bottom,
                label=pillar, color=palette[i % len(palette)],
                alpha=0.85, width=0.85)
        bottom = [b + v for b, v in zip(bottom, vals)]

    ax1.set_ylabel("# Pillar-Targeting Events", color="#c9cdd6", fontsize=11)
    ax1.set_title(
        f"Activist → Pillar Targeting Distribution  [{version_label}]",
        color="#ffffff", fontsize=14, pad=12, fontweight="bold",
    )
    ax1.legend(loc="upper right", framealpha=0.25, facecolor="#1a1d27",
               edgecolor="#2e3142", labelcolor="#c9cdd6", fontsize=8)
    ax1.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax1.grid(axis="y", color="#2e3142", linewidth=0.6)

    # Total line
    ax2.plot(steps, totals, color="#4e8ef7", linewidth=1.8,
             marker="o", markersize=2.5)
    ax2.fill_between(steps, totals, alpha=0.15, color="#4e8ef7")
    ax2.set_xlabel("Step", color="#c9cdd6", fontsize=11)
    ax2.set_ylabel("Total", color="#c9cdd6", fontsize=10)
    ax2.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax2.grid(axis="y", color="#2e3142", linewidth=0.6)

    plt.tight_layout(h_pad=0.5)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(output_path, dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"  Plot → {output_path}")


# ─────────────────────────────────────────────────────────────────────────────
# SUMMARY STATS
# ─────────────────────────────────────────────────────────────────────────────

def print_summary(records: list[dict]):
    counts: Counter = Counter()
    for rec in records:
        if rec.get("agent_type") == "ActivistAgent":
            pillar = _get_pillar(rec)
            if pillar:
                counts[pillar] += 1
    if not counts:
        print("  (no pillar-targeting events)")
        return
    width = max(len(p) for p in counts)
    for pillar, cnt in counts.most_common():
        print(f"    {pillar:<{width}}  {cnt:>6}")
    print(f"    {'TOTAL':<{width}}  {sum(counts.values()):>6}")


# ─────────────────────────────────────────────────────────────────────────────
# DISCOVER VERSIONED RUNS
# ─────────────────────────────────────────────────────────────────────────────

def discover_versions() -> list[tuple[Path, Path, str]]:
    """
    Scans DIRS["agent_dir"] for v<N> sub-directories and returns a list of
        (input_dir, output_dir, version_label)
    sorted numerically (v1, v2, v3, …).

    input_dir  — …/unstructured/v<N>   (must exist, contains step_*.txt files)
    output_dir — …/activists_logs/     (shared flat folder; files are named
                                        with a _v<N> suffix, no sub-folders)
    """
    input_base  = DIRS["agent_dir"]         # …/unstructured
    output_base = DIRS["agent_output_dir"]  # …/activists_logs  (flat, no v* sub-dirs)

    def _version_num(p: str) -> int:
        m = re.search(r"[/\\]v(\d+)$", p)
        return int(m.group(1)) if m else 0

    version_dirs = sorted(
        [p for p in glob.glob(str(input_base / "v*")) if os.path.isdir(p)],        
        key=_version_num,
    )

    if not version_dirs:
        raise FileNotFoundError(
            f"No versioned run folders (v1, v2, …) found under:\n  {input_base}\n"
            "Check that PROJECT_ROOT points to the right place."
        )

    runs = []
    for vdir in version_dirs:
        vname = Path(vdir).name   # "v1", "v2", …
        # All versions share the same output_base; the version is in the file name
        runs.append((Path(vdir), output_base, vname))

    return runs


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

def process_version(input_dir: Path, output_dir: Path, version_label: str):
    print(f"\n{'─'*60}")
    print(f"  Run: {version_label}")
    print(f"  Input  : {input_dir}")
    print(f"  Output : {output_dir}  (files suffixed _{version_label})")
    print(f"{'─'*60}")

    records = parse_all_steps(input_dir)

    csv_path  = output_dir / OUTPUT_CSV_TPL.format(v=version_label)
    plot_path = output_dir / OUTPUT_PLOT_TPL.format(v=version_label)

    write_csv(records, csv_path)
    plot_pillar_targeting(records, plot_path, version_label)

    print("  Pillar-targeting summary:")
    print_summary(records)


if __name__ == "__main__":
    runs = discover_versions()
    print(f"Discovered {len(runs)} version(s): "
          f"{', '.join(v for _, _, v in runs)}")

    for input_dir, output_dir, version_label in runs:
        process_version(input_dir, output_dir, version_label)

    print("\n✓ All runs processed.")

Discovered 5 version(s): v1, v2, v3, v4, v5

────────────────────────────────────────────────────────────
  Run: v1
  Input  : c:\Users\fmahnoor1\Documents\Research-FL\ABM-Repos\New Approach\pyabm-refactor\SimulationLogs\gemini_flash\ZeroShot\stateful\vision-4\AgentsLogging\simulation_log\unstructured\v1
  Output : c:\Users\fmahnoor1\Documents\Research-FL\ABM-Repos\New Approach\pyabm-refactor\SimulationLogs\gemini_flash\ZeroShot\stateful\vision-4\AgentsLogging\simulation_log\structured\activists_logs  (files suffixed _v1)
────────────────────────────────────────────────────────────
  Found 96 step files — parsing …
  Parsed 1,520 log records.
  CSV  → c:\Users\fmahnoor1\Documents\Research-FL\ABM-Repos\New Approach\pyabm-refactor\SimulationLogs\gemini_flash\ZeroShot\stateful\vision-4\AgentsLogging\simulation_log\structured\activists_logs\activists_logs_v1.csv
  Plot → c:\Users\fmahnoor1\Documents\Research-FL\ABM-Repos\New Approach\pyabm-refactor\SimulationLogs\gemini_flash\ZeroShot\stat

In [1]:


"""
Simulation Log Parser + Pillar-Targeting Distribution Plot
===========================================================
Automatically discovers every versioned run folder (v1, v2, v3, …).

Directory layout
────────────────
INPUT  (one folder per run):
    <PROJECT_ROOT>/SimulationLogs/gemini_flash/ZeroShot/stateful/vision-4/
        AgentsLogging/simulation_log/unstructured/v<N>/step_*.txt

OUTPUT (mirrored, one folder per run):
    <PROJECT_ROOT>/SimulationLogs/gemini_flash/ZeroShot/stateful/vision-4/
        AgentsLogging/simulation_log/structured/activists_logs/v<N>/
            parsed_simulation_log.csv
            pillar_targeting_distribution.png

Usage
─────
    python parse_and_plot_logs.py
"""

import ast
import csv
import glob
import json
import os
import re
from collections import Counter, defaultdict
from pathlib import Path

# ─────────────────────────────────────────────────────────────────────────────
# DIRECTORY CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

PROJECT_ROOT = Path.cwd().parent

_LOGS_BASE = (
    PROJECT_ROOT
    / "SimulationLogs" / "gemini_flash" / "ZeroShot"
    / "stateful"       / "vision-4"
    / "AgentsLogging"  / "simulation_log"
)

# Base directories — NO v* here; version suffix is appended at runtime
DIRS = {
    # Root that contains v1/, v2/, v3/, … sub-directories of raw step logs
    "agent_dir":        _LOGS_BASE / "unstructured",

    # Root where structured output folders v1/, v2/, … will be created
    "agent_output_dir": _LOGS_BASE / "structured" / "activists_logs",
}

# File name templates — {v} is replaced with the version label at runtime
# All files land directly in DIRS["agent_output_dir"] with no sub-folders.
OUTPUT_CSV_TPL  = "activists_logs_{v}.csv"
OUTPUT_PLOT_TPL = "pillar_targeting_distribution_{v}.png"


# ─────────────────────────────────────────────────────────────────────────────
# REGEX PATTERNS
# ─────────────────────────────────────────────────────────────────────────────

FIELD_PATTERN = re.compile(
    r"""
    (?P<key>[a-zA-Z_]+)
    =
    (?P<value>
        "(?:[^"\\]|\\.)*"       # double-quoted string
        |'(?:[^'\\]|\\.)*'      # single-quoted string
        |\([^)]*\)               # tuple  (x, y)
        |[^;{]+                  # plain value
    )
    """,
    re.VERBOSE,
)

CONTEXT_PATTERN = re.compile(r"context=(\{.*\})$", re.DOTALL)
MEMORY_PATTERN  = re.compile(r"memory=(\{.*?\}(?:\}|\s*\}))", re.DOTALL)


# ─────────────────────────────────────────────────────────────────────────────
# PARSING HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def _clean(v: str) -> str:
    v = v.strip()
    if (v.startswith('"') and v.endswith('"')) or \
       (v.startswith("'") and v.endswith("'")):
        return v[1:-1]
    return v


def _parse_tuple(v: str):
    try:
        return ast.literal_eval(v.strip())
    except Exception:
        return v.strip()


def parse_line(line: str) -> dict | None:
    line = line.strip()
    if not line:
        return None

    record = {}

    # 1. Peel off context={...} JSON blob
    ctx_match = CONTEXT_PATTERN.search(line)
    if ctx_match:
        try:
            record["context"] = json.loads(ctx_match.group(1))
        except json.JSONDecodeError:
            record["context_raw"] = ctx_match.group(1)
        line = line[: ctx_match.start()]

    # 2. Peel off memory={...} Python-dict blob
    mem_match = MEMORY_PATTERN.search(line)
    if mem_match:
        try:
            record["memory"] = ast.literal_eval(mem_match.group(1))
        except Exception:
            record["memory_raw"] = mem_match.group(1)
        line = line[: mem_match.start()]

    # 3. Parse remaining key=value fields
    for m in FIELD_PATTERN.finditer(line):
        record[m.group("key")] = _clean(m.group("value"))

    if not record:
        return None

    # 4. Type coercions
    for k in ("Step", "agent_id", "response_tokens"):
        if k in record:
            try:
                record[k] = int(record[k])
            except (ValueError, TypeError):
                pass

    if "elapsed" in record:
        try:
            record["elapsed"] = float(str(record["elapsed"]).rstrip("s"))
        except (ValueError, TypeError):
            pass

    for k in ("pos", "from_pos", "to_pos"):
        if k in record:
            record[k] = _parse_tuple(record[k])

    return record


def parse_step_file(filepath: str, step_number: int) -> list[dict]:
    records = []
    with open(filepath, "r", encoding="utf-8", errors="replace") as fh:
        for line in fh:
            rec = parse_line(line)
            if rec:
                rec.setdefault("Step", step_number)
                records.append(rec)
    return records


def _step_num(path: str) -> int:
    m = re.search(r"step_(\d+)\.txt$", path, re.IGNORECASE)
    return int(m.group(1)) if m else 0


def parse_all_steps(unstructured_dir: Path) -> list[dict]:
    """Parse every step_*.txt inside *unstructured_dir*."""
    files = sorted(
        glob.glob(str(unstructured_dir / "step_*.txt")),
        key=_step_num,
    )
    if not files:
        raise FileNotFoundError(
            f"No step_*.txt files found in:\n  {unstructured_dir}"
        )
    print(f"  Found {len(files)} step files — parsing …")
    records = []
    for fp in files:
        records.extend(parse_step_file(fp, _step_num(fp)))
    print(f"  Parsed {len(records):,} log records.")
    return records


# ─────────────────────────────────────────────────────────────────────────────
# CSV EXPORT
# ─────────────────────────────────────────────────────────────────────────────

FLAT_COLUMNS = [
    "Step", "agent_type", "agent_id", "event",
    "pos", "from_pos", "to_pos",
    "elapsed", "response_tokens",
    "target_agent", "target_pillar_inst", "primary_attribute",
    "citizen_cta", "coordination_state", "derived_strategy",
    "target_pillar_institution",
]


def _flatten(rec: dict) -> dict:
    flat = {col: rec.get(col, "") for col in FLAT_COLUMNS}

    ctx = rec.get("context", {})
    if isinstance(ctx, dict):
        flat["coordination_state"]       = flat["coordination_state"]       or ctx.get("coordination_state", "")
        flat["derived_strategy"]         = flat["derived_strategy"]         or ctx.get("derived_strategy", "")
        flat["target_pillar_institution"]= flat["target_pillar_institution"] or ctx.get("target_pillar_institution", "")

    # Stringify any leftover complex types
    for k in FLAT_COLUMNS:
        if isinstance(flat[k], (list, tuple, dict)):
            flat[k] = str(flat[k])

    return flat


def write_csv(records: list[dict], output_path: Path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=FLAT_COLUMNS, extrasaction="ignore")
        writer.writeheader()
        for rec in records:
            writer.writerow(_flatten(rec))
    print(f"  CSV  → {output_path}")


# ─────────────────────────────────────────────────────────────────────────────
# PLOT
# ─────────────────────────────────────────────────────────────────────────────

def _get_pillar(rec: dict) -> str:
    """Return the targeted pillar name from a record (checks multiple fields)."""
    pillar = rec.get("target_pillar_inst", "").strip()
    if not pillar:
        ctx = rec.get("context", {})
        if isinstance(ctx, dict):
            pillar = ctx.get("target_pillar_institution", "").strip()
    return pillar


def plot_pillar_targeting(records: list[dict], output_path: Path, version_label: str):
    try:
        import matplotlib.pyplot as plt
        import matplotlib.ticker as ticker
    except ImportError:
        print("  matplotlib not installed — pip install matplotlib")
        return

    step_total     : dict[int, int]             = defaultdict(int)
    step_breakdown : dict[int, dict[str, int]]  = defaultdict(lambda: defaultdict(int))

    for rec in records:
        if rec.get("agent_type") != "ActivistAgent":
            continue
        pillar = _get_pillar(rec)
        if pillar:
            s = rec.get("Step", 0)
            step_total[s] += 1
            step_breakdown[s][pillar] += 1

    if not step_total:
        print("  No pillar-targeting events found — skipping plot.")
        return

    steps   = sorted(step_total)
    totals  = [step_total[s] for s in steps]
    pillars = sorted({p for d in step_breakdown.values() for p in d})

    palette = [
        "#4e8ef7", "#f76e6e", "#6ef7b4", "#f7c06e", "#c06ef7",
        "#6ec9f7", "#f76ec0", "#c9f76e", "#f7906e", "#6e90f7",
    ]

    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(16, 9),
        gridspec_kw={"height_ratios": [3, 1.2]},
        sharex=True,
    )
    fig.patch.set_facecolor("#0f1117")
    for ax in (ax1, ax2):
        ax.set_facecolor("#1a1d27")
        ax.tick_params(colors="#c9cdd6")
        for spine in ax.spines.values():
            spine.set_edgecolor("#2e3142")

    # Stacked bars
    bottom = [0] * len(steps)
    for i, pillar in enumerate(pillars):
        vals = [step_breakdown[s].get(pillar, 0) for s in steps]
        ax1.bar(steps, vals, bottom=bottom,
                label=pillar, color=palette[i % len(palette)],
                alpha=0.85, width=0.85)
        bottom = [b + v for b, v in zip(bottom, vals)]

    ax1.set_ylabel("# Pillar-Targeting Events", color="#c9cdd6", fontsize=11)
    ax1.set_title(
        f"Activist → Pillar Targeting Distribution  [{version_label}]",
        color="#ffffff", fontsize=14, pad=12, fontweight="bold",
    )
    ax1.legend(loc="upper right", framealpha=0.25, facecolor="#1a1d27",
               edgecolor="#2e3142", labelcolor="#c9cdd6", fontsize=8)
    ax1.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax1.grid(axis="y", color="#2e3142", linewidth=0.6)

    # Total line
    ax2.plot(steps, totals, color="#4e8ef7", linewidth=1.8,
             marker="o", markersize=2.5)
    ax2.fill_between(steps, totals, alpha=0.15, color="#4e8ef7")
    ax2.set_xlabel("Step", color="#c9cdd6", fontsize=11)
    ax2.set_ylabel("Total", color="#c9cdd6", fontsize=10)
    ax2.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax2.grid(axis="y", color="#2e3142", linewidth=0.6)

    plt.tight_layout(h_pad=0.5)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(output_path, dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"  Plot → {output_path}")


# ─────────────────────────────────────────────────────────────────────────────
# SUMMARY STATS
# ─────────────────────────────────────────────────────────────────────────────

def print_summary(records: list[dict]):
    counts: Counter = Counter()
    for rec in records:
        if rec.get("agent_type") == "ActivistAgent":
            pillar = _get_pillar(rec)
            if pillar:
                counts[pillar] += 1
    if not counts:
        print("  (no pillar-targeting events)")
        return
    width = max(len(p) for p in counts)
    for pillar, cnt in counts.most_common():
        print(f"    {pillar:<{width}}  {cnt:>6}")
    print(f"    {'TOTAL':<{width}}  {sum(counts.values()):>6}")


# ─────────────────────────────────────────────────────────────────────────────
# DISCOVER VERSIONED RUNS
# ─────────────────────────────────────────────────────────────────────────────

def _vname_to_num(name: str) -> int:
    """'v1' -> 1, 'v21' -> 21, anything else -> 0."""
    m = re.match(r"^v(\d+)$", name, re.IGNORECASE)
    return int(m.group(1)) if m else 0


def discover_versions() -> list[tuple[Path, Path, str]]:
    """
    Scans DIRS["agent_dir"] for v<N> sub-directories using Path.iterdir()
    (more reliable than glob on Windows) and returns a list of
        (input_dir, output_dir, version_label)
    sorted numerically (v1, v2, v3, …).

    input_dir  — …/unstructured/v<N>   (must exist, contains step_*.txt files)
    output_dir — …/activists_logs/     (shared flat folder; files are named
                                        with a _v<N> suffix, no sub-folders)
    """
    input_base  = DIRS["agent_dir"]         # …/unstructured
    output_base = DIRS["agent_output_dir"]  # …/activists_logs (flat, no v* sub-dirs)

    if not input_base.exists():
        raise FileNotFoundError(
            f"Input directory does not exist:\n  {input_base}\n"
            "Check that PROJECT_ROOT points to the right place."
        )

    # Use iterdir() — works correctly on Windows regardless of path separators
    version_dirs = sorted(
        [p for p in input_base.iterdir() if p.is_dir() and _vname_to_num(p.name) > 0],
        key=lambda p: _vname_to_num(p.name),
    )

    print(f"  Scanning : {input_base}")
    print(f"  Found {len(version_dirs)} version folder(s): "
          f"{[p.name for p in version_dirs]}")

    if not version_dirs:
        raise FileNotFoundError(
            f"No v<N> sub-directories found under:\n  {input_base}\n"
            "Expected folders named v1, v2, v3, …"
        )

    return [
        (vdir, output_base, vdir.name)
        for vdir in version_dirs
    ]


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

def process_version(input_dir: Path, output_dir: Path, version_label: str):
    print(f"\n{'─'*60}")
    print(f"  Run: {version_label}")
    print(f"  Input  : {input_dir}")
    print(f"  Output : {output_dir}  (files suffixed _{version_label})")
    print(f"{'─'*60}")

    records = parse_all_steps(input_dir)

    csv_path  = output_dir / OUTPUT_CSV_TPL.format(v=version_label)
    plot_path = output_dir / OUTPUT_PLOT_TPL.format(v=version_label)

    write_csv(records, csv_path)
    plot_pillar_targeting(records, plot_path, version_label)

    print("  Pillar-targeting summary:")
    print_summary(records)


if __name__ == "__main__":
    runs = discover_versions()
    print(f"Discovered {len(runs)} version(s): "
          f"{', '.join(v for _, _, v in runs)}")

    for input_dir, output_dir, version_label in runs:
        process_version(input_dir, output_dir, version_label)

    print("\n✓ All runs processed.")


  Scanning : c:\Users\fmahnoor1\Documents\Research-FL\ABM-Repos\New Approach\pyabm-refactor\SimulationLogs\gemini_flash\ZeroShot\stateful\vision-4\AgentsLogging\simulation_log\unstructured
  Found 30 version folder(s): ['v1', 'v2', 'v3', 'v4', 'v5', 'v6', 'v7', 'v8', 'v9', 'v10', 'v11', 'v12', 'v13', 'v14', 'v15', 'v16', 'v17', 'v18', 'v19', 'v20', 'v21', 'v22', 'v23', 'v24', 'v25', 'v26', 'v27', 'v28', 'v29', 'v30']
Discovered 30 version(s): v1, v2, v3, v4, v5, v6, v7, v8, v9, v10, v11, v12, v13, v14, v15, v16, v17, v18, v19, v20, v21, v22, v23, v24, v25, v26, v27, v28, v29, v30

────────────────────────────────────────────────────────────
  Run: v1
  Input  : c:\Users\fmahnoor1\Documents\Research-FL\ABM-Repos\New Approach\pyabm-refactor\SimulationLogs\gemini_flash\ZeroShot\stateful\vision-4\AgentsLogging\simulation_log\unstructured\v1
  Output : c:\Users\fmahnoor1\Documents\Research-FL\ABM-Repos\New Approach\pyabm-refactor\SimulationLogs\gemini_flash\ZeroShot\stateful\vision-4\Agents